# 15. 10차 사전 등록 (김유환): 규제를 튜닝한 함께 학습 대비 UME, 상위 12개 지표 (새 테스트 주 s6)

`configs/prereg_yh_tuned.yaml` 대로 실행한다. s6 분할 만들기(처음 한 번, 약 10분) → 튜닝 54회(약 20분) → 학습 18회 + B2(약 8분) → 분석(약 15분). 모든 단계는 중단 후 다시 실행하면 이어서 진행한다.

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 준비물 · 사전 등록 확인 + s6 분할 만들기 (없을 때만, 약 10분)
from msrs_paper.config import load_yaml
from msrs_paper import data_prep as dp
from msrs_paper.utils import read_json, verify_manifest, write_manifest, now
m = verify_manifest(paths)
prereg = load_yaml("prereg_yh_tuned")
assert prereg["status"] == "final", "prereg_yh_tuned.yaml 확정 · 커밋이 필요합니다"
if not (paths.split_dir("s6") / "stats.json").exists():
    tx = dp.load_transactions(paths.raw / "transactions_train.csv")
    items = dp.load_items(paths).sort_values("idx").reset_index(drop=True)
    history, vt, tt, stats = dp.build_split(tx, items, cfg["data"]["window_weeks"], cfg["data"]["min_user_purchases"], 6)
    dp.save_split(paths.split_dir("s6"), history, vt, tt, stats)
    del tx
    m = write_manifest(paths)
    print("s6 분할 생성, 체크섬 목록 갱신:", len(m["files"]), "개 파일")
stats = read_json(paths.split_dir("s6") / "stats.json")
print(f"준비물 {len(m['files'])}개 일치 · 사전 등록 {prereg['registered']}")
print({k: v for k, v in stats.items() if k != "test_cells"})
log_file = paths.logs / "yh_tuned.log"
def log(msg):
    print(msg)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"{now()} {msg}\n")

In [ ]:
# @title 2. 튜닝 54회: C0 · C1 · C2 × 규제 격자 18 (s6 검증 주, seed 42, 약 20분)
import time
from msrs_paper import analysis_yh8
t0 = time.time()
tuning, selected = analysis_yh8.tune(cfg, exps, paths, log=log)
log(f"튜닝 완료 ({(time.time() - t0) / 60:.1f}분) · 선택 {selected}")
tuning.pivot_table(index="hp", columns="cond", values="valid_recall@300").round(4)

In [ ]:
# @title 3. 학습 18회 (기본 · 튜닝 설정 × C0 · C1 · C2 × seed 3, 벡터 저장) + B2 (약 8분)
t0 = time.time()
analysis_yh8.train(cfg, exps, paths, log=log)
log(f"10차 학습 완료 ({(time.time() - t0) / 60:.1f}분)")

In [ ]:
# @title 4. 분석: 확증 H20 · H21 · H22 + 탐색 (약 15분)
confirmatory, table, contrasts, hits, memo = analysis_yh8.run(cfg, exps, paths, log=log)
c = contrasts[contrasts["segment"] == "all"]
c[c["split"] == "s6"][["description", "metric", "value_a", "value_b", "estimate", "ci_low", "ci_high", "status"]]

In [ ]:
# @title 5. 탐색: 지표 표 · 결합 방식 · 적중 분해 · 외우기 정도 · 다른 주
display(table[table["segment"] == "all"].pivot_table(index="series", columns=["split", "metric"], values="value").round(4))
display(c[c["description"].str.startswith("결합")].pivot_table(index=["split", "description"], columns="metric", values="estimate").round(4))
display(hits.set_index(["split", "K"]).T.round(3))
display(memo.groupby(["setting", "cond"])["gap"].mean().unstack().round(4))
display(c[c["split"] != "s6"][["split", "description", "metric", "estimate", "ci_low", "ci_high"]])